#Rotaciones de Givens

In [19]:
import numpy as np

def givens(x1,x2):
    c = 1.
    s = 0.
    ax1 = abs(x1)
    ax2 = abs(x2)
    if ax1 + ax2 > 0:
        if ax2 > ax1:
            tau = -x1/x2
            s = -np.sign(x2)/np.sqrt(1 + tau**2)
            c = tau*s
        else:
            tau = -x2/x1
            c = np.sign(x1)/np.sqrt(1 + tau**2)
            s = tau*c
    return c, s

def qrgivens(A):
    m, n = A.shape
    Q = np.eye(m)
    R = A.copy()
    p = min(m - 1, n)

    for j in range(p):
        for i in range(j+1, m):
            if R[i, j] != 0:
                #I = [j, i], J = j:
                c, s = givens(R[j, j], R[i, j])
                G = np.array([[c, -s], [s, c]])

                R[[j, i], j:] = G @ R[[j, i], j:]
                Q[:, [j, i]] = Q[:, [j, i]] @ G.T

    if m <= n and R[m - 1, m - 1] < 0:
        # J = m:
        R[m - 1, m - 1:] = -R[m - 1, m - 1:]
        Q[:, m - 1] = -Q[:, m - 1]

    return Q, R


#TEST DE QRGIVENS
A = np.random.random((4, 5))
Q, R = qrgivens(A)
print(np.linalg.norm(A - Q @ R))

3.65923871021723e-16


#Reflexiones de Householder

In [22]:
import numpy as np

#Mejore la funcion holder para que sea mas facil
#Holder nos permite encontrar una reflexión tal Qx = y, con Q = (I-rho * (u @ u.T)) ortogonal

def holder(x):

  n = len(x)
  # Defino los parametros de salida
  rho = 0
  u = x.copy()
  u[0] = 1.

  # si n==1 tomamos sigma=0 y return u, rho definidos previamente
  if n == 1:
    sigma = 0
  # caso contrario definimos sigma como la suma de los elementos de x al cuadrado exceptuando x[0]
  else:
    sigma = np.sum(x[1:]**2)

  # Si sigma es mayor a cero entramos al siguiente if caso contrario vamos al return
  if sigma>0 or x[0]<0:
    #A partir de aqui el algoritmo es igual a de notas
    mu = np.sqrt(x[0]**2 + sigma)
    if x[0]<=0:
      gamma = x[0] - mu
    else:
      gamma = -sigma/(x[0] + mu)

    rho = 2*gamma**2/(gamma**2 + sigma)
    u = u/gamma
    u[0] = 1

  return u, rho

# #TEST
# x = np.random.random(4)
# u, rho = holder(x)
# Q = np.eye(4)- rho*np.outer(u,u)
# print(Q.T@Q)


def qrhholder(A): #Usando holder construimos Q ortogonal y R triangular superior con r_ii>0 tal que A=Q@R
    m, n = A.shape[0], A.shape[1]
    Q = np.eye(m)
    R = A.copy()

    p = min(m,n)

    for j in range(p):
        u, rho = holder(R[j:, j])
        w = rho*u
        R[j:, j:] = R[j:, j:] - np.outer(w, u.T@R[j:, j:])
        Q[:, j:] = Q[:, j:] - Q[:, j:] @ np.outer(w,u)

    return Q, R

A = np.random.random((3,5))

Q, R = qrhholder(A)

#print(Q)
#print(R)

print(np.linalg.norm(Q@R-A, 2))

5.41974423485201e-16
